# Real NVP — Density estimation using Real NVP (2016)

_Papers · Generative Models_

**Build a generative model out of invertible layers, so you can compute the exact likelihood of any data point and sample from the model — both cheaply.**

---

This notebook is a practice scaffold for the **Real NVP — Density estimation using Real NVP (2016)** lesson. We build it up one step at a time: a single affine coupling layer by hand, then a stack of them, then training by exact maximum likelihood. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Check one affine coupling layer by hand

The whole flow is built from **affine coupling layers**. Each layer splits the input in two: the first part is copied through unchanged, and the second part is scaled by `exp(s)` and shifted by `t`, where `s` and `t` are computed from the *copied* part. Because the copied part is untouched, the transform is trivially invertible (Eq. 9) and its Jacobian is triangular, so the log-determinant is just the sum of the scales `s` (Eq. 8).

We start by working one layer by hand on a single point to see Eqs. 7–9 in action.

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)

# One affine coupling layer by hand (Eqs. 7-9).
s_val = torch.tensor(0.5)    # scale returned by the s net
t_val = torch.tensor(-1.0)   # shift returned by the t net

x1 = torch.tensor(2.0)       # x = (2.0, 3.0); split at d=1
x2 = torch.tensor(3.0)

# Forward: data -> latent.
y1 = x1                              # Eq. 7: copy the first part unchanged
y2 = x2 * torch.exp(s_val) + t_val   # Eq. 7: scale-and-shift the second part

logdet = s_val                       # Eq. 8: log|det J| = sum_j s_j

# Inverse: latent -> data, recovering x2.
x2_back = (y2 - t_val) * torch.exp(-s_val)   # Eq. 9

print("forward y:", round(y1.item(), 6), round(y2.item(), 6))   # 2.0  3.946164
print("layer log-det:", round(logdet.item(), 6))               # 0.5
print("inverse recovers x2:", round(x2_back.item(), 6))        # 3.0

### Step 2 — Define the data, the coupling layer, and the stack

Now we generalise. `sample_moons` draws points from a two-moons distribution — a 2-D shape that a plain Gaussian can't capture. The `Coupling` module is the layer from Step 1 written with neural nets for `s` and `t`, where a binary `mask` chooses which coordinate is copied (`mask == 1`) and which is transformed (`mask == 0`). `RealNVP` stacks several couplings, **alternating** the mask each layer so that every coordinate eventually gets transformed.

In [ ]:
# --- A 2-D toy distribution: the two-moons shape. ---
def sample_moons(n):
    k = torch.randint(0, 2, (n,))        # which moon each point belongs to
    th = torch.rand(n) * math.pi         # angle along the crescent
    x = torch.stack([th.cos(), th.sin()], 1)
    # Second moon: flip and offset.
    x[k == 1] = torch.stack([1 - th[k == 1].cos(), 0.5 - th[k == 1].sin()], 1)
    return (x + torch.randn(n, 2) * 0.05) * 2.0


# --- A small MLP used for both the s (scale) and t (shift) nets. ---
def mlp(out):
    return nn.Sequential(
        nn.Linear(2, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, out),
    )


# --- The affine coupling layer. mask picks which coords are copied. ---
class Coupling(nn.Module):
    def __init__(self, mask):              # mask: 1 = copy this coord, 0 = transform it
        super().__init__()
        self.register_buffer("mask", mask)
        self.s_net = mlp(2)
        self.t_net = mlp(2)

    def forward(self, x):                  # data -> latent (Eqs. 7-8)
        xa = x * self.mask                 # the untouched part feeds s, t
        s = self.s_net(xa) * (1 - self.mask)   # only the transformed coords get a scale
        t = self.t_net(xa) * (1 - self.mask)
        y = xa + (1 - self.mask) * (x * torch.exp(s) + t)
        return y, s.sum(1)                 # Eq. 8: log-det = sum of log-scales

    def inverse(self, y):                  # latent -> data (Eq. 9)
        ya = y * self.mask
        s = self.s_net(ya) * (1 - self.mask)
        t = self.t_net(ya) * (1 - self.mask)
        x = ya + (1 - self.mask) * ((y - t) * torch.exp(-s))
        return x


# --- Stack couplings, alternating the mask each layer. ---
class RealNVP(nn.Module):
    def __init__(self, n_layers=6, alternate=True):
        super().__init__()
        layers = []
        for i in range(n_layers):
            if alternate and i % 2 == 0 or not alternate:
                m = torch.tensor([1.0, 0.0])
            else:
                m = torch.tensor([0.0, 1.0])
            layers.append(Coupling(m))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):                  # returns z and total log-det
        ld = torch.zeros(x.size(0))
        for layer in self.layers:
            x, d = layer(x)
            ld = ld + d
        return x, ld

    def inverse(self, z):
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z


# --- The base density and the exact log-likelihood. ---
def base_logpdf(z):                        # standard Gaussian N(0, I), D=2
    return -0.5 * (z ** 2).sum(1) - math.log(2 * math.pi)

def loglik(model, x):                      # Eq. 1: exact change-of-variables log-likelihood
    z, ld = model(x)
    return base_logpdf(z) + ld

### Step 3 — Train by exact maximum likelihood, then sample and ablate

Because the flow is invertible with a cheap Jacobian, the change-of-variables formula (Eq. 1) gives the **exact** log-likelihood of each point — so we can train by directly maximising it (minimising the negative log-likelihood). After training we check the held-out likelihood, verify the inverse really reconstructs the input, and draw new samples by pushing Gaussian noise *backward* through the stack.

The ablation at the end freezes the mask (no alternation). Without alternation the first coordinate is never transformed, so it stays a plain Gaussian and the likelihood gets worse — confirming why alternating masks matter.

In [ ]:
# --- Train by EXACT maximum likelihood (minimize negative log-likelihood). ---
def train(alternate=True, steps=4000):
    torch.manual_seed(0)
    model = RealNVP(n_layers=6, alternate=alternate)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for step in range(steps):
        x = sample_moons(512)
        nll = -loglik(model, x).mean()     # negative log-likelihood
        opt.zero_grad()
        nll.backward()
        opt.step()
        if step % 1000 == 0:
            print(f"  step {step:4d}  log-likelihood/point {(-nll).item():.3f}")
    return model

print("TRAIN (alternating masks):")
model = train(alternate=True)

# Exact log-likelihood on fresh data, and a round-trip invertibility check.
xt = sample_moons(2000)
print("\nexact mean log-likelihood (held-out):", round(loglik(model, xt).mean().item(), 3))

z, _ = model(xt)
x_rt = model.inverse(z)
print("max round-trip reconstruction error:", round((x_rt - xt).abs().max().item(), 8))  # ~0 (exact inverse)

# SAMPLE: draw z ~ N(0,I), run the stack backward.
z = torch.randn(2000, 2)
samples = model.inverse(z)
print("sample mean/std:",
      [round(v, 3) for v in samples.mean(0).tolist()],
      [round(v, 3) for v in samples.std(0).tolist()])
# (Our small run -- not the paper's reported number.)

# --- ABLATION: freeze the mask (no alternation), retrain. Log-likelihood gets worse. ---
print("\nABLATION (no mask alternation -> coordinate 1 never transformed):")
model_fixed = train(alternate=False)
print("ablated mean log-likelihood (held-out):", round(loglik(model_fixed, xt).mean().item(), 3))
# Worse: with a fixed mask the first coordinate stays a plain Gaussian and cannot fit the moons.

## Visualize the data & results

_Does maximizing the exact change-of-variables log-likelihood teach the flow to map a standard Gaussian onto the two-moons distribution, so that samples land on the moons?_

### Step 1 — Train a fresh flow for the visualization

This is a self-contained copy of the model and training loop (so the cell runs on its own). We rebuild `sample_moons`, the coupling layer, and the stack with alternating masks, then train for the same 4000 steps on the exact log-likelihood (Eq. 1).

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)

def sample_moons(n):
    k = torch.randint(0, 2, (n,))
    th = torch.rand(n) * math.pi
    x = torch.stack([th.cos(), th.sin()], 1)
    x[k == 1] = torch.stack([1 - th[k == 1].cos(), 0.5 - th[k == 1].sin()], 1)
    return (x + torch.randn(n, 2) * 0.05) * 2.0

def mlp(out):
    return nn.Sequential(
        nn.Linear(2, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, out),
    )

class Coupling(nn.Module):                 # affine coupling layer (Eqs. 7-9)
    def __init__(self, mask):
        super().__init__()
        self.register_buffer("mask", mask)
        self.s_net = mlp(2)
        self.t_net = mlp(2)
    def forward(self, x):
        xa = x * self.mask
        s = self.s_net(xa) * (1 - self.mask)
        t = self.t_net(xa) * (1 - self.mask)
        y = xa + (1 - self.mask) * (x * torch.exp(s) + t)
        return y, s.sum(1)                 # Eq. 8 log-det
    def inverse(self, y):
        ya = y * self.mask
        s = self.s_net(ya) * (1 - self.mask)
        t = self.t_net(ya) * (1 - self.mask)
        return ya + (1 - self.mask) * ((y - t) * torch.exp(-s))   # Eq. 9

class RealNVP(nn.Module):
    def __init__(self, n=6):
        super().__init__()
        masks = [torch.tensor([1.0, 0.0]), torch.tensor([0.0, 1.0])]
        self.layers = nn.ModuleList([Coupling(masks[i % 2]) for i in range(n)])
    def forward(self, x):
        ld = torch.zeros(x.size(0))
        for l in self.layers:
            x, d = l(x)
            ld = ld + d
        return x, ld
    def inverse(self, z):
        for l in reversed(self.layers):
            z = l.inverse(z)
        return z

model = RealNVP()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(4000):                      # train on exact log-likelihood (Eq. 1)
    x = sample_moons(512)
    z, ld = model(x)
    nll = -(-0.5 * (z ** 2).sum(1) - math.log(2 * math.pi) + ld).mean()
    opt.zero_grad()
    nll.backward()
    opt.step()

### Step 2 — Push a Gaussian blob through the inverse flow

To sample, we start from a round cloud of latent points `z ~ N(0, I)` and run them *backward* through the trained flow. If training worked, those Gaussian points come out landing on the two-moons shape — the model has learned to bend a simple Gaussian into the data distribution.

In [ ]:
torch.manual_seed(7)

z = torch.randn(60, 2)             # latent cloud: a round Gaussian blob
samples = model.inverse(z)         # push through the inverse flow -> data space

print("latent  :", [[round(p[0].item(), 2), round(p[1].item(), 2)] for p in z[:5]], "...")
print("samples :", [[round(p[0].item(), 2), round(p[1].item(), 2)] for p in samples[:5]], "...")
# Latents are a round Gaussian blob; samples land on the two-moons shape.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your stacked Real NVP fits the two-moons distribution well. Now make every
            coupling layer use the same mask (copy coordinate 1, transform coordinate 2, in every
            layer) instead of alternating. What happens to the fit, and what does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace the alternating masks with a single fixed mask repeated in all layers; keep depth, the $s,t$ network sizes, optimizer, data, and steps identical. — _An honest ablation changes exactly one thing &mdash; the mask alternation &mdash; so any difference is attributable to it._
- Retrain and check the held-out log-likelihood and samples. The first coordinate is now never transformed by the flow, so its marginal stays a plain standard Gaussian. — _With a fixed mask, coordinate 1 is copied through every layer; the flow can only ever reshape coordinate 2 as a function of coordinate 1._
- Conclude that alternation is what lets every coordinate be transformed, which is required to model a distribution where both coordinates have non-Gaussian, coupled structure (like two moons). — _Real NVP relies on composing layers with swapped masks (&sect;3.5) so the whole input is eventually transformed._

**Answer:** The fit collapses: the model's first coordinate stays a standard Gaussian (it is copied
                 unchanged through every layer), so it cannot capture the two-moons shape, and the log-likelihood
                 is much worse. Because the only change was removing mask alternation, this isolates
                 alternating masks as essential &mdash; a single coupling layer transforms only half the
                 coordinates, so you must swap which half is copied to eventually transform them all. (The CODE
                 includes this ablation.)

</details>

**Problem 2.** Using the worked layer ($s = 0.5$, $t = -1.0$, split $d=1$), push the point $x = (0.0,\ 1.0)$
            forward through one coupling layer. What is $y$, and what is this layer's log-determinant?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Copy the first part: $y_1 = x_1 = 0.0$. — _Eq. 7 leaves $x_{1:d}$ unchanged: $y_{1:d} = x_{1:d}$._
- Scale-and-shift the second part: $y_2 = x_2\,\exp(s) + t = 1.0 \times \exp(0.5) + (-1.0) = 1.648721 - 1.0 = 0.648721$. — _Eq. 7: $y_{d+1:D} = x_{d+1:D}\odot\exp(s) + t$, with $\exp(0.5)\approx 1.648721$._
- Log-det: $\sum_j s_j = 0.5$. — _Eq. 8: the Jacobian is triangular with diagonal $\exp(s)$, so $\log|\det J| = \sum_j s_j$ (one transformed coordinate here)._

**Answer:** $y = (0.0,\ 0.648721)$ and the layer's log-determinant is $0.5$. The first coordinate passed
                 through untouched; the second was scaled by $\exp(0.5)\approx1.65$ then shifted by $-1$. The
                 log-det is just the sum of the log-scales &mdash; no $2\times2$ determinant needed &mdash;
                 which is the whole point of the coupling design.

</details>

**Problem 3.** Why does Real NVP give the exact log-likelihood while a VAE only gives a lower bound? Both
            map data to a Gaussian latent space.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that Real NVP's map $f$ is a deterministic bijection: every $x$ has exactly one $z$ and vice versa, related by the closed-form inverse. — _A bijection lets you apply the change-of-variables formula (Eq. 1) directly, which is an exact equality, not an inequality._
- Recall a VAE's encoder is stochastic and its decoder is a separate network &mdash; there is no exact inverse, so the true likelihood $p(x)=\int p(x\mid z)p(z)\,dz$ is an intractable integral. — _The VAE cannot evaluate that integral, so it maximizes a tractable lower bound (the ELBO) instead._
- See that Real NVP avoids the integral entirely: because $f$ is invertible with a cheap (triangular) Jacobian, Eq. 1 evaluates $\log p_X(x)$ in one forward pass. — _No latent integral, no bound &mdash; just the exact change-of-variables value._

**Answer:** Real NVP's transform is an exact, invertible bijection with a tractable Jacobian determinant,
                 so the change-of-variables formula (Eq. 1) gives $\log p_X(x)$ exactly in one pass. A
                 VAE's encoder/decoder are not inverses of each other, so its true likelihood is an intractable
                 integral over the latent variable; it can only maximize a lower bound (the ELBO). Exact
                 likelihood is the payoff of insisting on an invertible, cheap-Jacobian map.

</details>